In [ ]:
import calendar
import pandas as pd
import os
from functools import reduce
from io import BytesIO

import ipywidgets as widgets
from openpyxl.styles import PatternFill
from IPython.display import display, clear_output

In [ ]:
# ESTATÍSTICAS ====================================================================
# Chaves obrigatórias por entrada:
#   label        – texto exibido no checkbox
#   monthly_col  – nome da coluna gerada em Mensal_Por_Ano
#   monthly_src  – coluna-fonte no df diário para Mensal_Por_Ano
#   hist_m_cols  – lista de tuplas (nome_coluna, coluna-fonte, função) para Historico_Mensal
#   hist_y_cols  – lista de tuplas (nome_coluna, coluna-fonte, função) para Historico_Anual
#                  lista vazia [] para omitir da tabela. None para herdar legado.
STAT_CATALOG = {
    'media': {
        'label':       'Média',
        'monthly_col': 'Média Diária',
        'monthly_src': 'Média Diária',
        'hist_m_cols': [
            ('Média dos Totais Mensais',  'Total Mensal',  'mean'),
            ('Média Diária Histórica',    'Média Diária',  'mean'),
        ],
        'hist_y_cols': [
            ('Média dos Totais Mensais',  'Total Mensal',  'mean'),
            ('Média Diária Anual',        'Média Diária',  'mean'),
        ],
    },
    'desvio': {
        'label':       'Desvio Padrão',
        'monthly_col': 'Desvio Padrão Diário',
        'monthly_src': 'Desvio Padrão Diário',
        'hist_m_cols': [
            ('Desvio Padrão dos Totais',  'Total Mensal',   'std'),
        ],
        'hist_y_cols': [
            ('Desvio Padrão dos Totais',  'Total Mensal',   'std'),
        ],
    },
    'mediana': {
        'label':       'Mediana',
        'monthly_col': 'Mediana Diária',
        'monthly_src': 'Mediana Diária',
        'hist_m_cols': [
            ('Mediana dos Totais Mensais','Total Mensal',   'median'),
            ('Mediana Diária Histórica',  'Mediana Diária', 'mean'),
        ],
        'hist_y_cols': [
            ('Mediana dos Totais Mensais','Total Mensal',   'median'),
            ('Mediana Diária Anual',      'Mediana Diária', 'mean'),
        ],
    },
    'minimo': {
        'label':       'Mínimo',
        'monthly_col': 'Mínimo Diário',
        'monthly_src': 'Mínimo Diário',
        'hist_m_cols': [
            ('Mínimo dos Totais Mensais', 'Total Mensal',   'min'),
            ('Mínimo Diário Histórico',   'Mínimo Diário',  'min'),
        ],
        'hist_y_cols': [
            ('Mínimo dos Totais Mensais', 'Total Mensal',   'min'),
            ('Mínimo Diário Anual',       'Mínimo Diário',  'min'),
        ],
    },
    'maximo': {
        'label':       'Máximo',
        'monthly_col': 'Máximo Diário',
        'monthly_src': 'Máximo Diário',
        'hist_m_cols': [
            ('Máximo dos Totais Mensais', 'Total Mensal',   'max'),
            ('Máximo Diário Histórico',   'Máximo Diário',  'max'),
        ],
        'hist_y_cols': [
            ('Máximo dos Totais Mensais', 'Total Mensal',   'max'),
            ('Máximo Diário Anual',       'Máximo Diário',  'max'),
        ],
    },
    'amplitude': {
        'label':       'Amplitude',
        'monthly_col': 'Amplitude Diária',
        'monthly_src': 'Amplitude Diária',
        'hist_m_cols': [
            ('Amplitude dos Totais',      'Total Mensal', lambda x: x.max() - x.min()),
            ('Amplitude Diária Histórica','Amplitude Diária', 'mean'),
        ],
        'hist_y_cols':  [],
    },
    'assimetria': {
        'label':       'Assimetria',
        'monthly_col': None,
        'monthly_src': None,
        'hist_m_cols': [('Assimetria Histórica', 'Total Mensal', 'skew')],
        'hist_y_cols': [('Assimetria Mensal',    'Total Mensal', 'skew')],
    },
    'dias_chuva_0': {
        'label':       'Dias c/ Chuva (> 0mm)',
        'monthly_col': 'Dias > 0mm',
        'monthly_src': 'Dias > 0mm',
        'hist_m_cols': [('Total Dias > 0mm', 'Dias > 0mm', 'sum')],
        'hist_y_cols': [('Total Dias > 0mm', 'Dias > 0mm', 'sum')],
    },
    'dias_chuva_10': {
        'label':       'Dias c/ Chuva (> 10mm)',
        'monthly_col': 'Dias > 10mm',
        'monthly_src': 'Dias > 10mm',
        'hist_m_cols': [('Total Dias > 10mm', 'Dias > 10mm', 'sum')],
        'hist_y_cols': [('Total Dias > 10mm', 'Dias > 10mm', 'sum')],
    },
    'dias_chuva_25': {
        'label':       'Dias c/ Chuva (> 25mm)',
        'monthly_col': 'Dias > 25mm',
        'monthly_src': 'Dias > 25mm',
        'hist_m_cols': [('Total Dias > 25mm', 'Dias > 25mm', 'sum')],
        'hist_y_cols': [('Total Dias > 25mm', 'Dias > 25mm', 'sum')],
    },
    'dias_chuva_50': {
        'label':       'Dias c/ Chuva (> 50mm)',
        'monthly_col': 'Dias > 50mm',
        'monthly_src': 'Dias > 50mm',
        'hist_m_cols': [('Total Dias > 50mm', 'Dias > 50mm', 'sum')],
        'hist_y_cols': [('Total Dias > 50mm', 'Dias > 50mm', 'sum')],
    },
}

# WIDGETS ==============================================================
upload_widget = widgets.FileUpload(
    accept='.csv', multiple=False,
    description='Selecionar arquivo .csv',
    layout=widgets.Layout(width='auto')
)

use_all_toggle = widgets.ToggleButtons(
    options=['Todos os dados', 'Definir intervalo'],
    value='Todos os dados',
    style={'description_width': '0px', 'button_width': '140px'}
)

year_start_widget = widgets.BoundedIntText(
    value=1900, min=1900, max=2100,
    description='De:', style={'description_width': '30px'},
    layout=widgets.Layout(width='130px')
)
year_end_widget = widgets.BoundedIntText(
    value=2026, min=1900, max=2100,
    description='Até:', style={'description_width': '30px'},
    layout=widgets.Layout(width='130px')
)
year_range_box = widgets.HBox(
    [year_start_widget, year_end_widget],
    layout=widgets.Layout(display='none', margin='4px 0 0 0')
)

threshold_widget = widgets.BoundedIntText(
    value=25, min=1, max=31,
    description='Dias mínimos válidos por mês:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='280px')
)

# Checkboxes gerados dinamicamente a partir do STAT_CATALOG
stat_checkboxes = {
    key: widgets.Checkbox(
        value=True,
        description=meta['label'],
        indent=False,
        layout=widgets.Layout(width='160px')
    )
    for key, meta in STAT_CATALOG.items()
}

run_button = widgets.Button(
    description=' Processar', button_style='primary', icon='play',
    layout=widgets.Layout(width='130px', height='34px')
)
download_widget = widgets.HTML(value='')
output_area     = widgets.Output()

# CALLBACKS ==============================================================
def on_toggle_change(change):
    year_range_box.layout.display = 'none' if change['new'] == 'Todos os dados' else 'flex'

use_all_toggle.observe(on_toggle_change, names='value')

def on_new_upload(change):
    if change['new']:
        output_buffer.clear()
        download_widget.value = ''
        with output_area:
            clear_output()

upload_widget.observe(on_new_upload, names='value')

# LAYOUT ================================================================
def _label(text):
    return widgets.HTML(
        f'<span style="font-weight:600;font-size:12px;color:#555;'
        f'text-transform:uppercase;letter-spacing:.5px">{text}</span>'
    )

def _divider():
    return widgets.HTML('<hr style="border:none;border-top:1px solid #e0e0e0;margin:6px 0">')

section_upload = widgets.VBox([
    _label('Arquivo'),
    upload_widget,
])

section_interval = widgets.VBox([
    _label('Período de análise'),
    use_all_toggle,
    year_range_box,
], layout=widgets.Layout(margin='0'))

section_quality = widgets.VBox([
    _label('Qualidade'),
    threshold_widget,
])

_cb_list = list(stat_checkboxes.values())

half = (len(_cb_list) + 1) // 2
left_column = widgets.VBox(_cb_list[:half])
right_column = widgets.VBox(_cb_list[half:])
_cb_container = widgets.HBox(
    [left_column, right_column], 
    layout=widgets.Layout(gap='10px')
)
section_stats = widgets.VBox([
    _label('Estatísticas a calcular'),
    _cb_container,
])

section_run = widgets.VBox([
    _label('Executar'),
    widgets.HBox([run_button, download_widget],
                 layout=widgets.Layout(align_items='center', gap='12px',)),
])

panel = widgets.VBox(
    [
        section_upload,
        _divider(),
        section_interval,
        _divider(),
        section_quality,
        _divider(),
        section_stats,
        _divider(),
        section_run,
        output_area,
    ],
    layout=widgets.Layout(
        padding='16px',
        border='1px solid #d0d0d0',
        border_radius='8px',
        max_width='520px',
        gap='10px',
    )
)

display(panel)

# Variável para guardar o arquivo gerado em memória
output_buffer = {}

def get_selected_stats():
    """Retorna lista de chaves do catálogo marcadas pelo usuário."""
    return [k for k, cb in stat_checkboxes.items() if cb.value]


# FUNÇÕES: CÁLCULOS COM SUFIXOS DE COLUNAS ===============================================
def calc_monthly_by_year(df_subset, suffix, selected_stats):
    if df_subset.empty:
        return pd.DataFrame(columns=['EstacaoCodigo', 'Ano', 'Mes'])

    # Colunas de qualidade
    keep    = ['EstacaoCodigo', 'Ano', 'Mes', 'Registros Válidos']
    rename  = {'Registros Válidos': f'Registros Válidos{suffix}'}

    # Total Mensal
    keep.append('Total Mensal')
    rename['Total Mensal'] = f'Total Mensal{suffix}'

    # Adiciona apenas as estatísticas selecionadas pelo usuário
    for key in selected_stats:
        meta = STAT_CATALOG[key]
        col  = meta['monthly_col']
        src  = meta['monthly_src']
        if src in df_subset.columns and src not in keep:
            keep.append(src)
            rename[src] = f'{col}{suffix}'

    result = df_subset[[c for c in keep if c in df_subset.columns]].copy()
    result = result.rename(columns=rename)
    return result

def calc_historical_monthly(df_subset, suffix, threshold, selected_stats):
    if df_subset.empty:
        return pd.DataFrame(columns=['EstacaoCodigo', 'Mes'])

    df_valid = df_subset[df_subset['Registros Válidos'] >= threshold]
    if df_valid.empty:
        return pd.DataFrame(columns=['EstacaoCodigo', 'Mes'])

    # Colunas de qualidade
    agg = {
        f'Anos Válidos{suffix}':            ('Total Mensal', 'count'),
        f'Total Registros Válidos{suffix}': ('Registros Válidos', 'sum'),
        f'Total Mensal Acumulado{suffix}':  ('Total Mensal', 'sum'),
    }
    # Acrescenta as estatísticas selecionadas que têm agregação histórica
    for key in selected_stats:
        for (col_name, src_col, fn) in STAT_CATALOG[key].get('hist_m_cols', []):
            agg[f'{col_name}{suffix}'] = (src_col, fn)

    result = df_valid.groupby(['EstacaoCodigo', 'Mes']).agg(**agg).reset_index()
    return result

def calc_historical_annual(df_subset, suffix, threshold, selected_stats):
    if df_subset.empty:
        return pd.DataFrame(columns=['EstacaoCodigo', 'Ano'])

    df_valid = df_subset[df_subset['Registros Válidos'] >= threshold]

    valid_months_per_year = (
        df_valid.groupby(['EstacaoCodigo', 'Ano'])['Mes']
        .count()
        .reset_index()
        .rename(columns={'Mes': 'Meses Válidos'})
    )

    # Colunas de qualidade
    agg = {
        f'Total Registros Válidos{suffix}': ('Registros Válidos', 'sum'),
        f'Total Acumulado{suffix}':         ('Total Mensal', 'sum'),
    }
    # Acrescenta as estatísticas selecionadas que têm agregação histórica anual
    for key in selected_stats:
        for (col_name, src_col, fn) in STAT_CATALOG[key].get('hist_y_cols', []):
            agg[f'{col_name}{suffix}'] = (src_col, fn)

    result = df_valid.groupby(['EstacaoCodigo', 'Ano']).agg(**agg).reset_index()
    
    if 'maximo' in selected_stats and 'minimo' in selected_stats:
        max_year = df_valid.groupby(['EstacaoCodigo', 'Ano'])['Máximo Diário'].max()
        min_year = df_valid.groupby(['EstacaoCodigo', 'Ano'])['Mínimo Diário'].min()
        result[f'Amplitude Anual{suffix}'] = (max_year - min_year).values

    result = result.merge(valid_months_per_year, on=['EstacaoCodigo', 'Ano'], how='left')
    result = result.rename(columns={'Meses Válidos': f'Meses Válidos{suffix}'})

    qual_cols  = ['EstacaoCodigo', 'Ano',
                  f'Total Registros Válidos{suffix}', f'Meses Válidos{suffix}']
    other_cols = [c for c in result.columns if c not in qual_cols]
    result = result[qual_cols + other_cols]
    return result

# LÓGICA PRINCIPAL ========================================================================
def on_run_clicked(b):
    output_buffer.clear()
    download_widget.value = ''

    with output_area:
        clear_output()

        # Verifica se arquivo foi carregado
        if not upload_widget.value:
            print("Nenhum arquivo carregado. Por favor, faça o upload de um CSV.")
            return

        if isinstance(upload_widget.value, dict):
            input_filename = list(upload_widget.value.keys())[0]
            uploaded_file  = list(upload_widget.value.values())[0]
            raw_bytes      = uploaded_file['content']
        else:
            uploaded_file  = upload_widget.value[0]
            input_filename = uploaded_file['name']
            raw_bytes      = bytes(uploaded_file['content'])

        # Lê o CSV considerando vírgulas como decimais
        df = pd.read_csv(BytesIO(raw_bytes), encoding='latin1', sep=';', skiprows=14, decimal=',')

        # EXCLUIR COLUNAS DE METADADOS
        rain_cols = [f'Chuva{i:02d}' for i in range(1, 32)]
        essential_cols = ['EstacaoCodigo', 'Data', 'NivelConsistencia']

        available_cols = [c for c in essential_cols + rain_cols if c in df.columns]
        df = df[available_cols].copy()

        date_column = 'Data'

        if date_column not in df.columns:
            print(f"AVISO: Coluna '{date_column}' não encontrada! Verifique o arquivo original.")
            return

        # Extrai Ano e Mês
        df['Data_dt'] = pd.to_datetime(df[date_column], format='%d/%m/%Y', errors='coerce')
        df['Ano'] = df['Data_dt'].dt.year
        df['Mes'] = df['Data_dt'].dt.month
        df = df.dropna(subset=['Ano', 'Mes'])

        # Converte Mês e Ano para inteiros (sem casas decimais)
        df['Ano'] = df['Ano'].astype(int)
        df['Mes'] = df['Mes'].astype(int)

        # Apaga a coluna original de data
        df = df.drop(columns=[date_column, 'Data_dt'])

        # SELEÇÃO DO INTERVALO DE ANÁLISE
        year_min = int(df['Ano'].min())
        year_max = int(df['Ano'].max())
        print(f"Dados disponíveis de {year_min} a {year_max}.")

        use_all = use_all_toggle.value == 'Todos os dados'
        if use_all:
            year_start, year_end = year_min, year_max
            print(f"Usando todos os dados ({year_min}–{year_max}, {len(df)} registros).")
        else:
            year_start = year_start_widget.value
            year_end   = year_end_widget.value

            if not (year_min <= year_start <= year_end <= year_max):
                print(f"Intervalo inválido. Use valores entre {year_min} e {year_max}, com início ≤ fim.")
                return

            df = df[(df['Ano'] >= year_start) & (df['Ano'] <= year_end)]
            print(f"Análise restrita a {year_start}–{year_end} ({len(df)} registros).")

        # LIMIAR DE QUALIDADE E ESTATÍSTICAS SELECIONADAS
        threshold      = threshold_widget.value
        selected_stats = get_selected_stats()

        if not selected_stats:
            print("Selecione ao menos uma estatística antes de processar.")
            return

        # Calcula as colunas base; as opcionais só se selecionadas
        rain_cols = [c for c in df.columns if (c.startswith('Chuva') and not c.endswith('Status'))]
        df['Total Mensal']      = df[rain_cols].sum(axis=1, skipna=True)
        df['Registros Válidos'] = df[rain_cols].notna().sum(axis=1)
        # Dias esperados no mês
        df['_days'] = df.apply(
            lambda r: calendar.monthrange(int(r['Ano']), int(r['Mes']))[1], axis=1
        )
        df['% Falhas'] = ((df['_days'] - df['Registros Válidos']) / df['_days'] * 100).round(1)
        df = df.drop(columns=['_days'])

        stat_compute = {
            'media':         lambda: df[rain_cols].mean(axis=1, skipna=True),
            'desvio':        lambda: df[rain_cols].std(axis=1, skipna=True),
            'mediana':       lambda: df[rain_cols].median(axis=1),
            'minimo':        lambda: df[rain_cols].min(axis=1),
            'maximo':        lambda: df[rain_cols].max(axis=1),
            'amplitude':     lambda: df[rain_cols].max(axis=1) - df[rain_cols].min(axis=1),
            'assimetria':    None,
            'dias_chuva_0':  lambda: (df[rain_cols] > 0).sum(axis=1),
            'dias_chuva_10': lambda: (df[rain_cols] > 10).sum(axis=1),
            'dias_chuva_25': lambda: (df[rain_cols] > 25).sum(axis=1),
            'dias_chuva_50': lambda: (df[rain_cols] > 50).sum(axis=1),
        }
        for key in selected_stats:
            fn = stat_compute.get(key)
            if fn is not None:
                col = STAT_CATALOG[key]['monthly_col']
                df[col] = fn()

        # SEPARAÇÃO DOS DADOS E APLICAÇÃO DOS CÁLCULOS
        df_raw        = df[df['NivelConsistencia'] == 1].drop_duplicates(subset=['EstacaoCodigo', 'Ano', 'Mes'])
        df_consistent = df[df['NivelConsistencia'] == 2].drop_duplicates(subset=['EstacaoCodigo', 'Ano', 'Mes'])
        df_all        = df.sort_values(['Registros Válidos', 'NivelConsistencia'], ascending=[False, False]).drop_duplicates(subset=['EstacaoCodigo', 'Ano', 'Mes'])

        # TABELA 2: MENSAL POR ANO
        dfs_ma = [d for d in [
            calc_monthly_by_year(df_all,        '',            selected_stats),
            calc_monthly_by_year(df_raw,        ' Bruto',      selected_stats),
            calc_monthly_by_year(df_consistent, ' Consistido', selected_stats)
        ] if not d.empty]

        if dfs_ma:
            monthly_by_year = (
                reduce(lambda l, r: pd.merge(l, r, on=['EstacaoCodigo', 'Ano', 'Mes'], how='outer'), dfs_ma)
                .sort_values(['Ano', 'Mes'], ascending=[False, False])
            )
            id_cols    = ['EstacaoCodigo', 'Ano', 'Mes']
            qual_names = ['Registros Válidos']
            stat_names = ['Total Mensal'] + [STAT_CATALOG[k]['monthly_col'] for k in selected_stats]
            suffixes  = ['', ' Bruto', ' Consistido']
            qual_cols = [f'{n}{s}' for n in qual_names for s in suffixes
                         if f'{n}{s}' in monthly_by_year.columns]
            stat_cols = [f'{n}{s}' for n in stat_names for s in suffixes
                         if f'{n}{s}' in monthly_by_year.columns]
            monthly_by_year = monthly_by_year[id_cols + qual_cols + stat_cols]
        else:
            monthly_by_year = pd.DataFrame()

        # TABELA 3: HISTÓRICO MENSAL
        dfs_hm = [d for d in [
            calc_historical_monthly(df_all,        '',            threshold, selected_stats),
            calc_historical_monthly(df_raw,        ' Bruto',      threshold, selected_stats),
            calc_historical_monthly(df_consistent, ' Consistido', threshold, selected_stats)
        ] if not d.empty]

        if dfs_hm:
            monthly_stats = (
                reduce(lambda l, r: pd.merge(l, r, on=['EstacaoCodigo', 'Mes'], how='outer'), dfs_hm)
                .sort_values(['Mes'], ascending=True)
            )
            id_cols    = ['EstacaoCodigo', 'Mes']
            qual_names = ['Anos Válidos', 'Total Registros Válidos', 'Total Mensal Acumulado']
            stat_names = [col for k in selected_stats for (col, _, _) in STAT_CATALOG[k].get('hist_m_cols', [])]
            suffixes  = ['', ' Bruto', ' Consistido']
            qual_cols = [f'{n}{s}' for n in qual_names for s in suffixes
                         if f'{n}{s}' in monthly_stats.columns]
            stat_cols = [f'{n}{s}' for n in stat_names for s in suffixes
                         if f'{n}{s}' in monthly_stats.columns]
            monthly_stats = monthly_stats[id_cols + qual_cols + stat_cols]
        else:
            monthly_stats = pd.DataFrame()

        # TABELA 4: HISTÓRICO ANUAL
        dfs_ha = [d for d in [
            calc_historical_annual(df_all,        '',            threshold, selected_stats),
            calc_historical_annual(df_raw,        ' Bruto',      threshold, selected_stats),
            calc_historical_annual(df_consistent, ' Consistido', threshold, selected_stats)
        ] if not d.empty]

        if dfs_ha:
            annual_stats = (
                reduce(lambda l, r: pd.merge(l, r, on=['EstacaoCodigo', 'Ano'], how='outer'), dfs_ha)
                .sort_values(['Ano'], ascending=False)
            )
            id_cols    = ['EstacaoCodigo', 'Ano']
            qual_names = ['Total Registros Válidos', 'Meses Válidos']
            stat_names = ['Total Acumulado'] + [col for k in selected_stats for (col, _, _) in STAT_CATALOG[k].get('hist_y_cols', [])]
            suffixes  = ['', ' Bruto', ' Consistido']
            qual_cols = [f'{n}{s}' for n in qual_names for s in suffixes
                         if f'{n}{s}' in annual_stats.columns]
            stat_cols = [f'{n}{s}' for n in stat_names for s in suffixes
                         if f'{n}{s}' in annual_stats.columns]
            annual_stats = annual_stats[id_cols + qual_cols + stat_cols]
        else:
            annual_stats = pd.DataFrame()
            
        # TABELA 5: FALHAS POR MÊS/ANO
        id_cols = ['EstacaoCodigo', 'Ano', 'Mes']
        faults_all  = (df[id_cols + ['% Falhas']]
            .groupby(id_cols, as_index=False)['% Falhas'].min())
        faults_raw  = (df[df['NivelConsistencia'] == 1][id_cols + ['% Falhas']]
            .groupby(id_cols, as_index=False)['% Falhas'].min()
            .rename(columns={'% Falhas': '% Falhas Bruto'}))
        faults_cons = (df[df['NivelConsistencia'] == 2][id_cols + ['% Falhas']]
            .groupby(id_cols, as_index=False)['% Falhas'].min()
            .rename(columns={'% Falhas': '% Falhas Consistido'}))
        faults_table = (faults_all
            .merge(faults_raw,  on=id_cols, how='outer')
            .merge(faults_cons, on=id_cols, how='outer')
            .sort_values(['Ano', 'Mes'], ascending=[False, False])
        )

        # REORGANIZAÇÃO DA ABA PRINCIPAL (Dados Completos)

        # Ordena os dados originais do mais recente para o mais antigo
        df = df.sort_values(['Ano', 'Mes', 'NivelConsistencia'], ascending=[False, False, False])

        # Define a nova ordem exata das colunas
        base_cols       = ['EstacaoCodigo', 'Ano', 'Mes', 'NivelConsistencia']
        calculated_cols = ['Total Mensal'] # Apenas o Total foi mantido
        new_order       = [col for col in base_cols + calculated_cols + rain_cols if col in df.columns]

        # Aplica a nova ordem e remove Média e Desvio Padrão do DataFrame principal
        df = df[new_order]

        # EXPORTAR PARA EXCEL (.xlsx) EM MEMÓRIA
        time_range      = f"{year_start}_{year_end}"
        base_name, _    = os.path.splitext(input_filename)
        output_filename = f"Calculos_{base_name}_{time_range}.xlsx"

        excel_buffer = BytesIO()
        with pd.ExcelWriter(excel_buffer, engine='openpyxl') as writer:
            df.to_excel(writer, sheet_name='Dados_Completos', index=False)
            if not monthly_by_year.empty:
                monthly_by_year.to_excel(writer, sheet_name='Mensal_Por_Ano',   index=False)
                monthly_stats.to_excel(  writer, sheet_name='Historico_Mensal', index=False)
                annual_stats.to_excel(   writer, sheet_name='Historico_Anual',  index=False)
                faults_table.to_excel(   writer, sheet_name='Falhas',           index=False)

            # Definição das cores (códigos HEX)
            fill_header     = PatternFill(start_color="D9D9D9", end_color="D9D9D9", fill_type="solid") # Cinza claro
            fill_bruto      = PatternFill(start_color="FCE4D6", end_color="FCE4D6", fill_type="solid") # Laranja claro
            fill_consistido = PatternFill(start_color="DDEBF7", end_color="DDEBF7", fill_type="solid") # Azul claro

            # Itera sobre todas as abas geradas
            for sheet_name in writer.sheets:
                sheet = writer.sheets[sheet_name]
                
                # Aplica cor ao cabeçalho (linha 1)
                for cell in sheet[1]:
                    cell.fill = fill_header

                # Itera sobre as colunas para aplicar cores nos dados
                for col_idx in range(1, sheet.max_column + 1):
                    header_value = str(sheet.cell(row=1, column=col_idx).value)
                    
                    # Verifica o sufixo da coluna para determinar a cor
                    if header_value.endswith('Bruto'):
                        for row_idx in range(2, sheet.max_row + 1):
                            sheet.cell(row=row_idx, column=col_idx).fill = fill_bruto
                            
                    elif header_value.endswith('Consistido'):
                        for row_idx in range(2, sheet.max_row + 1):
                            sheet.cell(row=row_idx, column=col_idx).fill = fill_consistido

        excel_buffer.seek(0)
        excel_data = excel_buffer.read()

        print(f"Arquivo pronto: {output_filename}")
        
        # GERAÇÃO DO LINK DE DOWNLOAD HTML
        import base64
        b64 = base64.b64encode(excel_data).decode()
        mime = 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'
        
        download_widget.value = f'''
            <a href="data:{mime};base64,{b64}" download="{output_filename}"
               style="text-decoration:none">
                <button class="jupyter-widgets jupyter-button widget-button mod-success"
                        style="width:130px;height:34px;font-size:13px;cursor:pointer">
                    <i class="fa fa-download"></i> Baixar Excel
                </button>
            </a>
        '''
        
run_button.on_click(on_run_clicked)